# Direct Classifier

This notebook runs in a Python virtual environment under WSL2. CUDA would not work for me on Windows natively, so I set the environment up in WSL2 and open it from Visual Studio. The main libraries are PyTorch, torchvision, and scikit-learn. CBIS-DDSM sits in the same folder as the .venv, in the folder structure it originally came in.

Architecture 5 is the one I kept, after four earlier attempts that are written up in the report. Stage 1 trains a single-view head, and stage 2 reuses that backbone on the CC and MLO pair. The patch notebook adds a Design B pretrain before stage 1, a three-class head on 300x300 crops, and copies only the backbone into stage 1. This notebook starts from ImageNet. From stage 2 both arms are the same two-view model: one shared EfficientNet-B3, the two feature maps concatenated, one MBConv block, and one logit. At test both arms see the full CC and MLO views.

I leave RUN_SWEEP and RUN_WINNING_ARM_TRAIN off, so a normal run scores the saved checkpoint and does not replace it.

![Training paths](figures/training_paths.png)

Patch pre-train versus direct classifier pipeline


In [1]:
import json
import os
import random
import time
import traceback
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B3_Weights, efficientnet_b3
from torchvision.models.efficientnet import MBConv, MBConvConfig

_SMOKE = os.environ.get("CM3070_SMOKE") == "1"
RUN_EVAL = False if _SMOKE else True
RUN_SWEEP = False
RUN_WINNING_ARM_TRAIN = False
REBUILD_DATA = False

def find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "project_specification.txt").is_file() and (cand / "notebooks").is_dir():
            return cand
    raise FileNotFoundError("open this notebook from the project folder")

REPO = find_repo()

def data_path(path) -> Path:
    p = Path(path)
    if p.is_file():
        return p
    rel = REPO / str(path).lstrip("/")
    if rel.is_file():
        return rel
    return p


CBIS_ROOT = REPO / "cbis-ddsm"
JPEG_DIR = CBIS_ROOT / "jpeg"
CSV_DIR = CBIS_ROOT / "csv"
FROZEN_DATA = REPO / "data"
REBUILD_DATA_DIR = REPO / "data_rebuild"
SINGLE_DATA_DIR = FROZEN_DATA
PAIRED_DATA_DIR = FROZEN_DATA / "paired"
WEIGHTS = REPO / "weights"
FROZEN_DC_ROOT = WEIGHTS / "direct_classifier"
FROZEN_DC_S1 = FROZEN_DC_ROOT / "stage1" / "best_val.pt"
FROZEN_DC_S2 = FROZEN_DC_ROOT / "stage2" / "best_val.pt"
TRAIN_ROOT = WEIGHTS / "direct_classifier_retrain"
FROZEN_PATCH_ROOT = WEIGHTS / "patch_classifier"
FROZEN_WRITE_ROOTS = (
    FROZEN_DATA,
    FROZEN_DC_ROOT,
    FROZEN_PATCH_ROOT,
    REPO / "demo" / "scores",
)

SEED = 42
VAL_FRACTION = 0.10
SINGLE_VAL_FRACTION = 0.05
HEIGHT = 576
WIDTH = 448
ARCHIVED_PLAIN = 0.8221767757382282
ARCHIVED_TTA = 0.8300578611332802

def check_saved_folder(path: Path) -> None:
    resolved = Path(path).resolve()
    for root in FROZEN_WRITE_ROOTS:
        root_resolved = root.resolve()
        if resolved == root_resolved or root_resolved in resolved.parents:
            raise RuntimeError(f"not writing over the saved results: {path}")


## Data Preparation

Stage 1 uses the single-image split: 2,674 training rows, 124 validation, and 689 test. The validation rows are a GroupShuffleSplit on patient ID, taken out of the official training split. Rows with no ROI mask are already out of these counts.

![Stage 1 splits](figures/stage1_splits.png)

Stage 1 data split breakdown

Stage 2 is the two-view split: 1,099 training exams, 122 validation, and 291 test. I pair the mass table and the calcification table separately, and I keep a pair only when the CC and MLO views are both present and the pathology matches. That drops 532 single-view rows. Training pathology is 456 malignant and 643 benign, and the test split is 112 malignant and 179 benign. A segmentation experiment cut the test set further, to 283 exams that had masks, but I did not use that cut.

![Stage 2 splits](figures/stage2_splits.png)

Stage 2 data split breakdown

Taking validation out of the training split still leaves one problem. CBIS-DDSM leaks patients between the official train and test split. I could rebuild that split so one patient never appears in both, but then I would not be using the official CBIS-DDSM split. So this project has the same limit Wang pointed out in her 2024 review of internal validation.

Stage 1 takes 5 percent of the official training rows as validation, grouped by patient ID, which is the 124 single-image rows. Stage 2 takes 10 percent, which is the 122 paired exams.

Two filters remove rows before training. Stage 1 drops 81 rows that have no region-of-interest mask. Stage 2 drops 532 single-view rows that do not have both CC and MLO with matching pathology.

I reused the JSONL from the midterm, with the same fixed seed, rather than drawing a new split or running K-fold. A single-image record stores the image path, a binary pathology flag, the patient ID, and a mask path when one exists.

![Loader](figures/loader.png)

Loader diagram


In [2]:
PATHOLOGY_MAP = {
    "MALIGNANT": "malignant",
    "BENIGN": "benign",
    "BENIGN_WITHOUT_CALLBACK": "benign",
}

def resolve_series_dir(csv_relpath: str) -> Path | None:
    # CSV path -> jpeg series folder
    for part in csv_relpath.strip().split("/"):
        candidate = JPEG_DIR / part
        if candidate.is_dir():
            return candidate
    return None

def resolve_image_path(csv_relpath: str) -> Path | None:
    series_dir = resolve_series_dir(csv_relpath)
    if series_dir is None:
        return None
    image_files = [
        f for f in series_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg")
    ]
    if len(image_files) != 1:
        return None
    return image_files[0]

def find_mask_path(mask_dir: Path | None, full_size: tuple[int, int]) -> Path | None:
    if mask_dir is None:
        return None
    for f in mask_dir.iterdir():
        if f.suffix.lower() not in (".jpg", ".jpeg"):
            continue
        with Image.open(f) as im:
            if im.size == full_size:
                return f
    return None

def mask_to_bbox(mask_path: Path) -> tuple[int, int, int, int] | None:
    arr = np.array(Image.open(data_path(mask_path)).convert("L"))
    ys, xs = np.where(arr > 127)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def write_jsonl(rows: list[dict], path: Path) -> None:
    check_saved_folder(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


def build_single_example(row, abnormality_type: str, drop_reasons: list) -> dict | None:
    # one view, with the mask path and the box
    pathology = PATHOLOGY_MAP.get(str(row["pathology"]).strip())
    if pathology is None:
        drop_reasons.append("unmapped_pathology")
        return None

    image_dir = resolve_series_dir(row["image file path"])
    if image_dir is None:
        drop_reasons.append("missing_image_dir")
        return None
    image_files = [
        f for f in image_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg")
    ]
    if len(image_files) != 1:
        drop_reasons.append("ambiguous_full_image")
        return None
    image_path = image_files[0]
    with Image.open(image_path) as im:
        width, height = im.size

    mask_dir = resolve_series_dir(row["ROI mask file path"])
    mask_path = find_mask_path(mask_dir, (width, height))
    if mask_path is None:
        alt_dir = resolve_series_dir(row["cropped image file path"])
        mask_path = find_mask_path(alt_dir, (width, height))
    if mask_path is None:
        drop_reasons.append("missing_mask")
        return None

    bbox = mask_to_bbox(mask_path)
    if bbox is None:
        drop_reasons.append("empty_mask")
        return None
    x1, y1, x2, y2 = bbox

    return {
        "patient_id": row["patient_id"],
        "abnormality_type": abnormality_type,
        "pathology": pathology,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "width": width,
        "height": height,
        "bbox": [x1, y1, x2, y2],
    }

def load_single_examples(csv_path: Path, abnormality_type: str, drop_reasons: list) -> list[dict]:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    examples = []
    for _, row in df.iterrows():
        ex = build_single_example(row, abnormality_type, drop_reasons)
        if ex is not None:
            examples.append(ex)
    return examples

def prepare_single_image_data(
    out_dir: Path = SINGLE_DATA_DIR,
    val_fraction: float = SINGLE_VAL_FRACTION,
    seed: int = SEED,
) -> dict:
    drop_reasons: list[str] = []
    train_rows = load_single_examples(
        CSV_DIR / "mass_case_description_train_set.csv", "mass", drop_reasons
    )
    train_rows += load_single_examples(
        CSV_DIR / "calc_case_description_train_set.csv", "calcification", drop_reasons
    )
    test_rows = load_single_examples(
        CSV_DIR / "mass_case_description_test_set.csv", "mass", drop_reasons
    )
    test_rows += load_single_examples(
        CSV_DIR / "calc_case_description_test_set.csv", "calcification", drop_reasons
    )

    patient_ids = [r["patient_id"] for r in train_rows]
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    train_idx, val_idx = next(splitter.split(train_rows, groups=patient_ids))
    train_split = [train_rows[i] for i in train_idx]
    val_split = [train_rows[i] for i in val_idx]

    train_patients = {r["patient_id"] for r in train_split}
    val_patients = {r["patient_id"] for r in val_split}
    assert not (train_patients & val_patients), "patient leakage between train/val"

    write_jsonl(train_split, out_dir / "train.jsonl")
    write_jsonl(val_split, out_dir / "val.jsonl")
    write_jsonl(test_rows, out_dir / "test.jsonl")

    summary = {
        "train": len(train_split),
        "val": len(val_split),
        "test": len(test_rows),
        "dropped_total": len(drop_reasons),
        "drop_reasons": dict(Counter(drop_reasons)),
        "train_pathology": dict(Counter(r["pathology"] for r in train_split)),
        "val_pathology": dict(Counter(r["pathology"] for r in val_split)),
        "test_pathology": dict(Counter(r["pathology"] for r in test_rows)),
    }
    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary

def pair_csv(csv_path: Path, abnormality_type: str, official_split: str):
    # pair CC and MLO in one CSV. Mass and calc stay separate.
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    groups: dict[tuple, dict] = defaultdict(dict)
    drops: list[str] = []

    for _, row in df.iterrows():
        raw = str(row["pathology"]).strip()
        pathology = PATHOLOGY_MAP.get(raw)
        if pathology is None:
            drops.append("unmapped_pathology")
            continue
        image_path = resolve_image_path(row["image file path"])
        if image_path is None:
            drops.append("missing_image")
            continue
        key = (
            row["patient_id"],
            str(row["left or right breast"]).strip().upper(),
            int(row["abnormality id"]),
        )
        view = str(row["image view"]).strip().upper()
        groups[key][view] = {
            "pathology": pathology,
            "pathology_raw": raw,
            "image_path": str(image_path),
        }

    pairs = []
    for (patient_id, breast, abn_id), views in groups.items():
        if "CC" in views and "MLO" in views:
            # BENIGN and BENIGN_WITHOUT_CALLBACK both count as benign.
            # Drop the pair if the raw pathology strings still differ.
            if views["CC"]["pathology_raw"] != views["MLO"]["pathology_raw"]:
                drops.append("inconsistent_pathology")
                continue
            pairs.append({
                "lesion_id": f"{patient_id}_{breast}_{abn_id}",
                "patient_id": patient_id,
                "breast": breast,
                "abnormality_id": abn_id,
                "abnormality_type": abnormality_type,
                "pathology": views["CC"]["pathology"],
                "cc_image_path": views["CC"]["image_path"],
                "mlo_image_path": views["MLO"]["image_path"],
                "official_split": official_split,
            })
        else:
            drops.append("single_view_dropped")
    return pairs, drops

def prepare_paired_data(
    out_dir: Path = PAIRED_DATA_DIR,
    val_fraction: float = VAL_FRACTION,
    seed: int = SEED,
) -> dict:
    train_pairs: list[dict] = []
    test_pairs: list[dict] = []
    drop_reasons: list[str] = []

    for csv_name, atype, split, dest in [
        ("mass_case_description_train_set.csv", "mass", "train", train_pairs),
        ("calc_case_description_train_set.csv", "calcification", "train", train_pairs),
        ("mass_case_description_test_set.csv", "mass", "test", test_pairs),
        ("calc_case_description_test_set.csv", "calcification", "test", test_pairs),
    ]:
        pairs, drops = pair_csv(CSV_DIR / csv_name, atype, split)
        dest.extend(pairs)
        drop_reasons.extend(drops)

    patient_ids = [p["patient_id"] for p in train_pairs]
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    train_idx, val_idx = next(splitter.split(train_pairs, groups=patient_ids))
    train_split = [train_pairs[i] for i in train_idx]
    val_split = [train_pairs[i] for i in val_idx]

    # official_split is not written into the JSONL.
    def strip_official(rows):
        out = []
        for r in rows:
            out.append({k: v for k, v in r.items() if k != "official_split"})
        return out

    train_split = strip_official(train_split)
    val_split = strip_official(val_split)
    test_split = strip_official(test_pairs)

    train_patients = {r["patient_id"] for r in train_split}
    val_patients = {r["patient_id"] for r in val_split}
    assert not (train_patients & val_patients), "patient leakage between paired train/val"

    write_jsonl(train_split, out_dir / "train.jsonl")
    write_jsonl(val_split, out_dir / "val.jsonl")
    write_jsonl(test_split, out_dir / "test.jsonl")

    summary = {
        "train": len(train_split),
        "val": len(val_split),
        "test": len(test_split),
        "dropped_total": len(drop_reasons),
        "drop_reasons": dict(Counter(drop_reasons)),
        "train_pathology": dict(Counter(r["pathology"] for r in train_split)),
        "val_pathology": dict(Counter(r["pathology"] for r in val_split)),
        "test_pathology": dict(Counter(r["pathology"] for r in test_split)),
    }
    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


def use_data_dirs():
    global SINGLE_DATA_DIR, PAIRED_DATA_DIR
    if REBUILD_DATA:
        SINGLE_DATA_DIR = REBUILD_DATA_DIR
        PAIRED_DATA_DIR = REBUILD_DATA_DIR / "paired"
        print("Building JSONL under", REBUILD_DATA_DIR)
        print(json.dumps(prepare_single_image_data(SINGLE_DATA_DIR, SINGLE_VAL_FRACTION, SEED), indent=2))
        print(json.dumps(prepare_paired_data(PAIRED_DATA_DIR, VAL_FRACTION, SEED), indent=2))
    else:
        SINGLE_DATA_DIR = FROZEN_DATA
        PAIRED_DATA_DIR = FROZEN_DATA / "paired"
        print("Reusing frozen JSONL under", FROZEN_DATA.relative_to(REPO))

use_data_dirs()
for split_name, directory, expected in (
    ("single", SINGLE_DATA_DIR, {"train": 2674, "val": 124, "test": 689}),
    ("paired", PAIRED_DATA_DIR, {"train": 1099, "val": 122, "test": 291}),
):
    counts = {}
    for split in ("train", "val", "test"):
        path = directory / f"{split}.jsonl"
        with path.open() as handle:
            counts[split] = sum(1 for line in handle if line.strip())
        if counts[split] != expected[split]:
            raise RuntimeError(f"{split_name} {split} has {counts[split]} rows, expected {expected[split]}")
    print(split_name, counts)


Reusing frozen JSONL under data
single {'train': 2674, 'val': 124, 'test': 689}
paired {'train': 1099, 'val': 122, 'test': 291}


## Model Architecture Design

Stage 1 for both arms is a single-view EfficientNet-B3. The torchvision feature extractor is followed by global average pooling, dropout of 0.5, a 256-unit dense layer with ReLU, dropout of 0.3, and one logit. Both dropout rates come from Chollet 2018, and I used the same rates for the midterm. A sigmoid is applied to the logit only when a probability is needed. Training uses BCEWithLogitsLoss on the logit, with the positive-class weight taken from the training split only.

Stage 2 calls the same feature extractor on the CC tensor and the MLO tensor. The weights are shared, so both views update one module. Each map has 1536 channels. Concatenated, that is 3072, and it passes through one MBConv block with expansion ratio 2, a 3x3 kernel, and stride 1. Pooling and the dense head follow stage 1, and the first dense layer is sized for the fused 3072 channels. The pair then gives one logit. The two images are not stacked into a six-channel input.

![Architecture 5](figures/architecture5.png)

Architecture 5 stack

The classifier class is SharedTwoViewClassifier. When pretrained is true, the constructor loads torchvision EfficientNet-B3 with ImageNet weights and keeps the convolutional features as self.shared_core = eff.features. That is how the direct classifier starts. The patch arm builds the same class with pretrained set to false, copies the Design B backbone into stage 1, and then loads that stage 1 checkpoint into stage 2. In both cases the stock ImageNet head is discarded. The three-class patch head is discarded as well, and it does not share parameters with the exam head. Both views use the one module, so each training step updates the same convolutions.

The forward method takes cc_pixel_values and mlo_pixel_values, each shaped (B, 3, H, W). Both go through self.shared_core, or through torch.utils.checkpoint while training if I need the memory. The maps are concatenated on the channel axis. One MBConv block then refines that 3072-channel tensor, with expand_ratio 2, a 3x3 kernel, and stride 1, so fusion does not change the spatial size. Global average pooling collapses it to a 3072-dimensional vector, which goes through dropout. A few more layers give one logit. I only turn that logit into a probability at prediction time. In training the logit goes straight into BCEWithLogitsLoss.

![Stage 2 forward](figures/stage2_forward.png)

Stage 2 forward pass


In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]

IMAGENET_STD = [0.229, 0.224, 0.225]

STAGE4_WEIGHT_DECAY = 0.01

STAGE4_DENSE_UNITS = 256

STAGE4_DROPOUT1 = 0.5

STAGE4_DROPOUT2 = 0.3

STAGE4_N_FUSION_BLOCKS = 1

STAGE4_NUM_WORKERS = 2

def compute_pos_weight(labels: list[float]) -> float:
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    return n_neg / n_pos if n_pos > 0 else 1.0

def compute_metrics(y_true, y_score, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics["specificity"] = float(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    metrics["confusion_matrix"] = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    if len(set(y_true.tolist())) > 1:
        metrics["auc_roc"] = float(roc_auc_score(y_true, y_score))
    return metrics

def set_train_mode(model: nn.Module, frozen_submodules: list[nn.Module]):
    model.train()
    for m in frozen_submodules:
        m.eval()

def single_view_transform(img, height, width, train=False, augment="light"):
    img = img.resize((width, height))
    if train:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if augment == "medium":
            angle = random.uniform(-15, 15)
            scale = random.uniform(0.85, 1.15)
            shear = random.uniform(-5, 5)
            img = TF.affine(img, angle=angle, translate=[0, 0], scale=scale, shear=[shear, 0.0])
        elif augment == "strong":
            angle = random.uniform(-10, 10)
            scale = random.uniform(0.9, 1.1)
            img = TF.affine(img, angle=angle, translate=[0, 0], scale=scale, shear=0.0)
    return TF.normalize(TF.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)

class SingleViewMammogramDataset(Dataset):
    def __init__(self, jsonl_path, height, width, train=False, augment="light"):
        with open(jsonl_path) as f:
            self.rows = [json.loads(line) for line in f]
        self.height = height
        self.width = width
        self.train = train
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        image = Image.open(data_path(row["image_path"])).convert("RGB")
        pixel_values = single_view_transform(
            image, self.height, self.width, self.train, self.augment
        )
        label = 1.0 if row["pathology"] == "malignant" else 0.0
        return pixel_values, label

def single_view_collate_fn(batch):
    images, labels = zip(*batch, strict=True)
    return {
        "pixel_values": torch.stack(images),
        "labels": torch.tensor(labels, dtype=torch.float32),
    }

def paired_transform(cc_img, mlo_img, height, width, train=False, augment="light"):
    cc_img = cc_img.resize((width, height))
    mlo_img = mlo_img.resize((width, height))
    if train:
        if random.random() < 0.5:
            cc_img = TF.hflip(cc_img)
            mlo_img = TF.hflip(mlo_img)
        if augment == "medium":
            angle = random.uniform(-15, 15)
            scale = random.uniform(0.85, 1.15)
            shear = random.uniform(-5, 5)
            cc_img = TF.affine(cc_img, angle=angle, translate=[0, 0], scale=scale, shear=[shear, 0.0])
            mlo_img = TF.affine(mlo_img, angle=angle, translate=[0, 0], scale=scale, shear=[shear, 0.0])
        elif augment == "strong":
            angle = random.uniform(-10, 10)
            scale = random.uniform(0.9, 1.1)
            cc_img = TF.affine(cc_img, angle=angle, translate=[0, 0], scale=scale, shear=0.0)
            mlo_img = TF.affine(mlo_img, angle=angle, translate=[0, 0], scale=scale, shear=0.0)
    cc_t = TF.normalize(TF.to_tensor(cc_img), IMAGENET_MEAN, IMAGENET_STD)
    mlo_t = TF.normalize(TF.to_tensor(mlo_img), IMAGENET_MEAN, IMAGENET_STD)
    return cc_t, mlo_t

class PairedMammogramDataset(Dataset):
    def __init__(self, jsonl_path, height, width, train=False, augment="light"):
        with open(jsonl_path) as f:
            self.rows = [json.loads(line) for line in f]
        self.height = height
        self.width = width
        self.train = train
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        cc_img = Image.open(data_path(row["cc_image_path"])).convert("RGB")
        mlo_img = Image.open(data_path(row["mlo_image_path"])).convert("RGB")
        cc_t, mlo_t = paired_transform(
            cc_img, mlo_img, self.height, self.width, self.train, self.augment
        )
        label = 1.0 if row["pathology"] == "malignant" else 0.0
        return cc_t, mlo_t, label

def paired_collate_fn(batch):
    cc_imgs, mlo_imgs, labels = zip(*batch, strict=True)
    return {
        "cc_pixel_values": torch.stack(cc_imgs),
        "mlo_pixel_values": torch.stack(mlo_imgs),
        "labels": torch.tensor(labels, dtype=torch.float32),
    }

class SingleViewEffNetClassifier(nn.Module):
    def __init__(self, pretrained=True, dense_units=256, dropout1=0.5, dropout2=0.3,
                 use_grad_checkpoint=False):
        super().__init__()
        eff = efficientnet_b3(
            weights=EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.backbone = eff.features
        self.use_grad_checkpoint = use_grad_checkpoint
        self.dropout1 = nn.Dropout(dropout1)
        self.fc1 = nn.Linear(1536, dense_units)
        self.dropout2 = nn.Dropout(dropout2)
        self.fc2 = nn.Linear(dense_units, 1)

    def forward(self, pixel_values):
        if self.use_grad_checkpoint and self.training:
            x = torch.utils.checkpoint.checkpoint(
                self.backbone, pixel_values, use_reentrant=False
            )
        else:
            x = self.backbone(pixel_values)
        x = torch.flatten(nn.functional.adaptive_avg_pool2d(x, 1), 1)
        x = self.dropout1(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        return self.fc2(x).squeeze(-1)

class SharedTwoViewClassifier(nn.Module):
    def __init__(self, pretrained=True, n_fusion_blocks=1, dense_units=256,
                 dropout1=0.5, dropout2=0.3, stochastic_depth_prob=0.2,
                 use_grad_checkpoint=False):
        super().__init__()
        eff = efficientnet_b3(
            weights=EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.shared_core = eff.features
        self.use_grad_checkpoint = use_grad_checkpoint
        fusion_channels = 2 * 1536
        fusion_cfg = MBConvConfig(
            expand_ratio=2, kernel=3, stride=1,
            input_channels=fusion_channels, out_channels=fusion_channels, num_layers=1,
        )
        self.fusion_blocks = nn.Sequential(*[
            MBConv(fusion_cfg, stochastic_depth_prob, nn.BatchNorm2d)
            for _ in range(n_fusion_blocks)
        ])
        self.dropout1 = nn.Dropout(dropout1)
        self.fc1 = nn.Linear(fusion_channels, dense_units)
        self.dropout2 = nn.Dropout(dropout2)
        self.fc2 = nn.Linear(dense_units, 1)

    def load_stage1_backbone(self, stage1_state_dict_backbone):
        self.shared_core.load_state_dict(stage1_state_dict_backbone)

    def _core(self, x):
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(
                self.shared_core, x, use_reentrant=False
            )
        return self.shared_core(x)

    def forward(self, cc_pixel_values, mlo_pixel_values):
        cc_feat = self._core(cc_pixel_values)
        mlo_feat = self._core(mlo_pixel_values)
        x = torch.cat([cc_feat, mlo_feat], dim=1)
        x = self.fusion_blocks(x)
        x = torch.flatten(nn.functional.adaptive_avg_pool2d(x, 1), 1)
        x = self.dropout1(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        return self.fc2(x).squeeze(-1)

def atomic_torch_save(obj, path: Path):
    check_saved_folder(path)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def append_history_jsonl(path: Path, row: dict):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")

def load_history_jsonl(path: Path) -> list[dict]:
    path = Path(path)
    if not path.is_file():
        return []
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def amp_dtype_and_scaler(device: str, use_amp: bool):
    if not use_amp or device != "cuda":
        return None, None, False
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16, None, True  # bf16: no GradScaler
    scaler = torch.cuda.amp.GradScaler()
    return torch.float16, scaler, True

def feasibility_probe(height, width, batch_size, two_view=False, use_amp=True,
                      n_steps=2, device="cuda"):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    if two_view:
        model = SharedTwoViewClassifier(pretrained=False).to(device)
    else:
        model = SingleViewEffNetClassifier(pretrained=False).to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    amp_dtype, scaler, amp_on = amp_dtype_and_scaler(device, use_amp)

    try:
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_steps):
            optimizer.zero_grad(set_to_none=True)
            if two_view:
                cc = torch.randn(batch_size, 3, height, width, device=device)
                mlo = torch.randn(batch_size, 3, height, width, device=device)
                y = torch.randint(0, 2, (batch_size,), device=device).float()
                with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_on):
                    logits = model(cc, mlo)
                    loss = loss_fn(logits, y)
            else:
                x = torch.randn(batch_size, 3, height, width, device=device)
                y = torch.randint(0, 2, (batch_size,), device=device).float()
                with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_on):
                    logits = model(x)
                    loss = loss_fn(logits, y)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        torch.cuda.synchronize()
        elapsed = (time.time() - t0) / n_steps
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        ok = True
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            ok = False
            elapsed, peak_gb = float("nan"), float("nan")
            torch.cuda.empty_cache()
        else:
            raise
    finally:
        del model, optimizer
        torch.cuda.empty_cache()
    return ok, elapsed, peak_gb

def choose_batch_size(height, width, two_view=False, use_amp=True, candidates=(4, 2, 1),
                      headroom_gb=2.0, device="cuda"):
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Probe {'two-view' if two_view else 'single-view'} @ {height}x{width} "
          f"(GPU {total_gb:.1f} GB, AMP={use_amp})")
    print(f"{'bs':>4} {'ok':>4} {'s/step':>8} {'peak_GB':>8}")
    best = None
    for bs in candidates:
        ok, elapsed, peak = feasibility_probe(
            height, width, bs, two_view=two_view, use_amp=use_amp, device=device
        )
        print(f"{bs:>4} {str(ok):>4} {elapsed:>8.3f} {peak:>8.2f}")
        if ok and peak <= total_gb - headroom_gb:
            best = bs
            break
        if ok and best is None:
            best = bs  # accept tight fit if nothing else works
    if best is None:
        raise RuntimeError(
            f"OOM at all batch sizes for {'two-view' if two_view else 'single-view'} "
            f"{height}x{width}"
        )
    return best

@torch.no_grad()
def evaluate_single(model, loader, device, frozen_submodules):
    model.eval()
    y_true, y_score = [], []
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        logits = model(pixel_values)
        y_score.extend(torch.sigmoid(logits).float().cpu().tolist())
        y_true.extend(batch["labels"].tolist())
    set_train_mode(model, frozen_submodules)
    return compute_metrics(y_true, y_score)

@torch.no_grad()
def evaluate_shared(model, loader, device, frozen_submodules):
    model.eval()
    y_true, y_score = [], []
    for batch in loader:
        cc = batch["cc_pixel_values"].to(device)
        mlo = batch["mlo_pixel_values"].to(device)
        logits = model(cc, mlo)
        y_score.extend(torch.sigmoid(logits).float().cpu().tolist())
        y_true.extend(batch["labels"].tolist())
    set_train_mode(model, frozen_submodules)
    return compute_metrics(y_true, y_score)


def make_stage2_model(pretrained: bool = False, device: str = "cpu") -> SharedTwoViewClassifier:
    model = SharedTwoViewClassifier(
        pretrained=pretrained,
        n_fusion_blocks=STAGE4_N_FUSION_BLOCKS,
        dense_units=STAGE4_DENSE_UNITS,
        dropout1=STAGE4_DROPOUT1,
        dropout2=STAGE4_DROPOUT2,
        use_grad_checkpoint=False,
    )
    return model.to(device)


def _extract_state(ckpt) -> dict:
    if isinstance(ckpt, dict) and "model" in ckpt:
        return ckpt["model"]
    return ckpt

def load_checkpoint(path: Path, model: nn.Module, device: str) -> dict:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {path}")
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(_extract_state(ckpt))
    return ckpt if isinstance(ckpt, dict) else {"model": ckpt}


@torch.no_grad()
def evaluate_with_tta(model, loader, device: str) -> dict:
    # plain ROC-AUC, plus a horizontal flip of both views
    model.eval()
    y_true, y_plain, y_tta = [], [], []
    for batch in loader:
        cc = batch["cc_pixel_values"].to(device)
        mlo = batch["mlo_pixel_values"].to(device)
        logits = model(cc, mlo)
        logits_f = model(torch.flip(cc, dims=[3]), torch.flip(mlo, dims=[3]))
        p = torch.sigmoid(logits).float().cpu()
        pf = torch.sigmoid(logits_f).float().cpu()
        y_plain.extend(p.tolist())
        y_tta.extend((0.5 * (p + pf)).tolist())
        y_true.extend(batch["labels"].tolist())
    plain = compute_metrics(y_true, y_plain)
    tta = compute_metrics(y_true, y_tta)
    return {"test_metrics": plain, "tta_metrics": tta, "n": len(y_true)}


def make_jpeg_paired_loaders(height=HEIGHT, width=WIDTH, batch_size=2, augment="light", num_workers=0):
    train_ds = PairedMammogramDataset(PAIRED_DATA_DIR / "train.jsonl", height, width, train=True, augment=augment)
    val_ds = PairedMammogramDataset(PAIRED_DATA_DIR / "val.jsonl", height, width, train=False)
    test_ds = PairedMammogramDataset(PAIRED_DATA_DIR / "test.jsonl", height, width, train=False)
    common = dict(collate_fn=paired_collate_fn, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, **common)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def score_frozen_direct_classifier(device=None, batch_size=2):
    if device is None:
        device = "cuda" if torch.cuda.is_available() and not _SMOKE else "cpu"
    model = make_stage2_model(pretrained=False, device=device)
    ckpt = load_checkpoint(FROZEN_DC_S2, model, device)
    _, _, test_loader, _, _, test_ds = make_jpeg_paired_loaders(batch_size=batch_size, num_workers=0)
    result = evaluate_with_tta(model, test_loader, device)
    result["checkpoint"] = str(FROZEN_DC_S2)
    result["n"] = len(test_ds)
    result["ckpt_phase"] = ckpt.get("phase") if isinstance(ckpt, dict) else None
    result["ckpt_epoch"] = ckpt.get("epoch") if isinstance(ckpt, dict) else None
    print(
        f"Direct Classifier JPEG n={result['n']} "
        f"plain={result['test_metrics']['auc_roc']:.4f} "
        f"TTA={result['tta_metrics']['auc_roc']:.4f} "
        f"(archived {ARCHIVED_PLAIN:.4f} / {ARCHIVED_TTA:.4f})"
    )
    return result


## Chollet grid

Stage 2 follows Chollet's freeze-then-fine-tune schedule, and the two arms do not share one timetable. The direct classifier freezes the convolutional core in phase 1 and trains the head for 5 epochs at 1e-3. Phase 2 unfreezes the network and fine-tunes for 20 epochs at 3e-5. That phase-2 learning rate, and these phase lengths, came out of the two-view hyperparameter sweep.

I tried six stage 1 arms at 576x448. The backbone stays frozen for phase 1, then I unfreeze it.

- s1_h_ctrl_p2_1e5: 5 epochs at 1e-3, then 20 at 1e-5, light augmentation
- s1_h_p2_3e5: 5 at 1e-3, then 20 at 3e-5, light. This is the arm I kept
- s1_h_p2_1e4: 5 at 1e-3, then 20 at 1e-4, light
- s1_h_p2_3e5_medaug: 5 at 1e-3, then 20 at 3e-5, medium augmentation
- s1_h_long_p2_3e5: 5 at 1e-3, then 35 at 3e-5, light
- s1_h_p1_10_p2_3e5: 10 at 1e-3, then 20 at 3e-5, light

Stage 2 for the arm I kept is s2_from_s1_h_p2_3e5.

On a pair, the same transform is applied to both views. The one I kept is a horizontal flip of CC and MLO together, so I do not flip one view and leave the other. Rotation of about 10 to 15 degrees, and zoom from 0.85 to 1.15, were tried on another run and did not help. I reuse the paired flip at test time and average the two probabilities. That flip is not used when I score the validation set.

The positive-class weight is the number of benign training examples divided by the number of malignant ones, taken from the training split only. For stage 2 that is 643 / 456. I pass it into BCEWithLogitsLoss (Chollet 2018).

The label is set before training from the pathology string, and the same rule is used for training, validation, and testing. Malignant is 1. Benign, including benign-without-callback, is 0. BI-RADS and breast density are not inputs. They can still be used afterwards to slice the results, which is also what Petrini and Kim do on CBIS-DDSM (Petrini and Kim 2025).

The optimizer is AdamW with weight decay 0.01. Phase 1 and phase 2 are separate functions. Phase 1 sets requires_grad false on the backbone. Phase 2 sets it true and fine-tunes end to end. Gradients accumulate across mini-batches, and the step is taken in mixed precision. After each epoch I set the model to eval, under torch.no_grad(), and scikit-learn computes the validation ROC-AUC. Loss and AUC are appended to history.jsonl.

The checkpoint and the hyperparameters were chosen on the validation split. The test split was not used to pick the epoch. Validation here is the 124 single-image rows and the 122 paired exams. I rank models by test ROC-AUC. Accuracy, sensitivity, specificity, precision, and F1 are still reported, but ROC-AUC is the one I follow. At a threshold of 0.5 the direct classifier has the higher accuracy and the higher specificity. I still keep ROC-AUC first, because the threshold can be moved later and ROC-AUC covers more than one operating point. The comparison notebook goes through that.

![Validation AUC](figures/val_auc.png)

Direct Classifier and Patch Arm Validation AUCs vs epochs


In [4]:

SWEEP_ROOT = TRAIN_ROOT
STATUS_PATH = SWEEP_ROOT / "sweep_status.json"
LEADERBOARD_PATH = SWEEP_ROOT / "leaderboard.json"
ARCHIVED_STAGE1_BUDGET_AUC = 0.8157
ARCHIVED_STAGE2_BUDGET_AUC = 0.8099
ARCHIVED_STAGE2_BUDGET_TTA_AUC = 0.8103

@dataclass
class ArmConfig:
    # one freeze-then-fine-tune arm

    arm_id: str
    stage: str  # "stage1" | "stage2"
    height: int
    width: int
    phase1_epochs: int
    phase2_epochs: int
    phase1_lr: float
    phase2_lr: float
    augment: str = "light"  # light | medium | strong
    target_effective_batch: int = 8
    use_grad_checkpoint: bool = False
    weight_decay: float = STAGE4_WEIGHT_DECAY
    seed: int = SEED
    # Stage2 only: which Stage1 arm to init from (uses that arm's best_val.pt)
    stage1_source: str | None = None
    # where this search writes its files
    priority: int = 100
    est_hours: float = 1.0
    notes: str = ""

    @property
    def out_dir(self) -> Path:
        return SWEEP_ROOT / self.arm_id

    def to_dict(self) -> dict:
        d = asdict(self)
        d["out_dir"] = str(self.out_dir)
        return d


def build_stage1_queue() -> list[ArmConfig]:
    # six stage 1 arms at 576x448. I kept s1_h_p2_3e5.
    half = (576, 448)
    arms: list[ArmConfig] = [
        ArmConfig(
            "s1_h_ctrl_p2_1e5", "stage1", *half, 5, 20, 1e-3, 1e-5,
            augment="light", priority=10, est_hours=0.9,
            notes="Control: budget-like Chollet at 576 with best_val selection",
        ),
        ArmConfig(
            "s1_h_p2_3e5", "stage1", *half, 5, 20, 1e-3, 3e-5,
            augment="light", priority=20, est_hours=0.9,
            notes="Higher phase-2 learning rate (3e-5). This is the arm I kept",
        ),
        ArmConfig(
            "s1_h_p2_1e4", "stage1", *half, 5, 20, 1e-3, 1e-4,
            augment="light", priority=30, est_hours=0.9,
            notes="Aggressive Phase-2 LR (1e-4), still after freeze",
        ),
        ArmConfig(
            "s1_h_p2_3e5_medaug", "stage1", *half, 5, 20, 1e-3, 3e-5,
            augment="medium", priority=40, est_hours=0.95,
            notes="Medium aug + mid Phase-2 LR",
        ),
        ArmConfig(
            "s1_h_long_p2_3e5", "stage1", *half, 5, 35, 1e-3, 3e-5,
            augment="light", priority=50, est_hours=1.4,
            notes="Longer Phase-2 (35) with mid LR; best_val guards overfit",
        ),
        ArmConfig(
            "s1_h_p1_10_p2_3e5", "stage1", *half, 10, 20, 1e-3, 3e-5,
            augment="light", priority=60, est_hours=1.1,
            notes="Longer Phase-1 head training before unfreeze",
        ),
    ]
    return sorted(arms, key=lambda a: a.priority)


def make_stage2_arm(source: ArmConfig, rank: int) -> ArmConfig:
    # same settings as stage 1, starting from that arm's best_val.pt
    return ArmConfig(
        arm_id=f"s2_from_{source.arm_id}",
        stage="stage2",
        height=source.height,
        width=source.width,
        phase1_epochs=source.phase1_epochs,
        phase2_epochs=max(15, min(source.phase2_epochs, 25)),
        phase1_lr=source.phase1_lr,
        phase2_lr=source.phase2_lr,
        augment=source.augment,
        target_effective_batch=source.target_effective_batch,
        use_grad_checkpoint=source.use_grad_checkpoint,
        weight_decay=source.weight_decay,
        seed=source.seed,
        stage1_source=source.arm_id,
        priority=200 + rank,
        est_hours=0.8,
        notes=f"Stage2 Chollet mirror of {source.arm_id}; init from best_val",
    )


def oom_fallback_variants(cfg: ArmConfig) -> list[ArmConfig]:
    # smaller setups if the first one runs out of memory
    variants = []
    base = deepcopy(cfg)
    # try gradient checkpointing first
    if not base.use_grad_checkpoint:
        v = deepcopy(base)
        v.arm_id = f"{cfg.arm_id}__ckpt"
        v.use_grad_checkpoint = True
        v.notes = (cfg.notes or "") + " [fallback: grad_ckpt]"
        variants.append(v)
    # then a smaller batch
    if base.target_effective_batch > 4:
        v = deepcopy(base)
        v.arm_id = f"{cfg.arm_id}__eb4"
        v.use_grad_checkpoint = True
        v.target_effective_batch = 4
        v.notes = (cfg.notes or "") + " [fallback: eff_batch=4]"
        variants.append(v)
    return variants


# scores saved as each arm finishes


def load_status() -> dict:
    if STATUS_PATH.is_file():
        with open(STATUS_PATH) as f:
            return json.load(f)
    return {
        "started_at": None,
        "deadline_ts": None,
        "completed": [],
        "failed": [],
        "skipped": [],
        "active": None,
        "stage2_planned": [],
    }


def save_status(status: dict):
    SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
    tmp = STATUS_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(status, f, indent=2)
    os.replace(tmp, STATUS_PATH)


def update_leaderboard(row: dict):
    board = []
    if LEADERBOARD_PATH.is_file():
        with open(LEADERBOARD_PATH) as f:
            board = json.load(f)
    board = [r for r in board if r.get("arm_id") != row.get("arm_id")]
    board.append(row)
    board.sort(key=lambda r: r.get("test_auc_best_val", 0.0) or 0.0, reverse=True)
    with open(LEADERBOARD_PATH, "w") as f:
        json.dump(board, f, indent=2)
    return board


# one arm: freeze the backbone, then fine-tune, and keep the best validation AUC


def _run_config_dict(cfg: ArmConfig, batch_size: int, accum_steps: int, use_amp: bool) -> dict:
    return {
        "arm_id": cfg.arm_id,
        "stage": cfg.stage,
        "height": cfg.height,
        "width": cfg.width,
        "phase1_epochs": cfg.phase1_epochs,
        "phase2_epochs": cfg.phase2_epochs,
        "phase1_lr": cfg.phase1_lr,
        "phase2_lr": cfg.phase2_lr,
        "augment": cfg.augment,
        "batch_size": batch_size,
        "accum_steps": accum_steps,
        "target_effective_batch": cfg.target_effective_batch,
        "use_amp": use_amp,
        "use_grad_checkpoint": cfg.use_grad_checkpoint,
        "weight_decay": cfg.weight_decay,
        "seed": cfg.seed,
        "stage1_source": cfg.stage1_source,
        "workflow": "chollet_freeze_then_ft",
        "selection": "global_best_val",
    }


def _configs_compatible(a: dict, b: dict) -> bool:
    keys = [
        "arm_id", "stage", "height", "width", "phase1_epochs", "phase2_epochs",
        "phase1_lr", "phase2_lr", "augment", "batch_size", "accum_steps",
        "use_amp", "use_grad_checkpoint", "seed", "stage1_source",
    ]
    return all(a.get(k) == b.get(k) for k in keys)


def resolve_resume(output_dir: Path, expected: dict, force_restart: bool = False):
    output_dir = Path(output_dir)
    if force_restart:
        for name in (
            "last.pt", "phase1_done.pt", "best_val.pt", "model.pt",
            "results.json", "test_results.json", "history.jsonl",
        ):
            p = output_dir / name
            if p.is_file():
                p.unlink()
        return None

    results = output_dir / "results.json"
    test_results = output_dir / "test_results.json"
    if results.is_file() and test_results.is_file():
        return {"status": "done"}

    last = output_dir / "last.pt"
    phase1_done = output_dir / "phase1_done.pt"
    if last.is_file():
        ckpt = torch.load(last, map_location="cpu", weights_only=False)
        if ckpt.get("config") and not _configs_compatible(ckpt["config"], expected):
            print(f"[resume] config mismatch in last.pt - cold start for {output_dir.name}")
            return None
        return {"status": "resume", "ckpt": ckpt, "source": "last.pt"}
    if phase1_done.is_file():
        ckpt = torch.load(phase1_done, map_location="cpu", weights_only=False)
        if ckpt.get("config") and not _configs_compatible(ckpt["config"], expected):
            return None
        return {"status": "phase1_done", "ckpt": ckpt, "source": "phase1_done.pt"}
    return None


def train_phase(
    *,
    model,
    train_loader,
    val_loader,
    device,
    epochs,
    lr,
    pos_weight,
    trainable_params,
    frozen_submodules,
    log_prefix,
    output_dir: Path,
    phase: str,
    run_config: dict,
    start_epoch: int = 1,
    history_so_far: list | None = None,
    global_best_auc: float = float("-inf"),
    weight_decay: float = 0.01,
    accum_steps: int = 1,
    use_amp: bool = True,
    mode: str = "single",
    optimizer=None,
    scheduler=None,
    scaler=None,
):
    # one phase. best_val is the best validation AUC so far
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    history = list(history_so_far or [])
    amp_dtype, new_scaler, amp_on = amp_dtype_and_scaler(device, use_amp)
    if scaler is None:
        scaler = new_scaler

    if optimizer is None:
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    if scheduler is None:
        last_epoch = start_epoch - 2 if start_epoch > 1 else -1
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, last_epoch=last_epoch
        )

    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    best_val_auc = global_best_auc

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        set_train_mode(model, frozen_submodules)
        running_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            labels = batch["labels"].to(device)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_on and device == "cuda"):
                if mode == "single":
                    logits = model(batch["pixel_values"].to(device))
                else:
                    logits = model(
                        batch["cc_pixel_values"].to(device),
                        batch["mlo_pixel_values"].to(device),
                    )
                loss = loss_fn(logits, labels) / accum_steps

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if step % accum_steps == 0 or step == len(train_loader):
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * accum_steps
            n_batches += 1

        scheduler.step()

        if mode == "single":
            val_metrics = evaluate_single(model, val_loader, device, frozen_submodules)
        else:
            val_metrics = evaluate_shared(model, val_loader, device, frozen_submodules)

        train_loss = running_loss / max(n_batches, 1)
        row = {
            "phase": phase,
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
        }
        history.append(row)
        append_history_jsonl(output_dir / "history.jsonl", row)

        val_auc = val_metrics.get("auc_roc", float("nan"))
        print(
            f"[{log_prefix}] {phase} epoch {epoch}/{epochs} "
            f"train_loss={train_loss:.4f} val_auc={val_auc:.4f} "
            f"val_sens={val_metrics['sensitivity']:.4f} "
            f"val_spec={val_metrics['specificity']:.4f} "
            f"lr={scheduler.get_last_lr()[0]:.2e} "
            f"elapsed={time.time() - t0:.0f}s",
            flush=True,
        )

        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "phase": phase,
            "epoch": epoch,
            "history": history,
            "config": run_config,
            "global_best_auc": best_val_auc,
        }
        atomic_torch_save(ckpt, output_dir / "last.pt")

        if val_auc == val_auc and val_auc > best_val_auc:
            best_val_auc = val_auc
            ckpt["global_best_auc"] = best_val_auc
            atomic_torch_save(ckpt, output_dir / "best_val.pt")
            print(f"[{log_prefix}]   up new global best_val auc={best_val_auc:.4f}", flush=True)

    return history, optimizer, scheduler, scaler, best_val_auc


def _load_stage1_best_backbone(stage1_arm_id: str, device: str) -> dict:
    # best_val.pt if it is there, otherwise model.pt
    s1_dir = SWEEP_ROOT / stage1_arm_id
    best = s1_dir / "best_val.pt"
    final = s1_dir / "model.pt"
    if best.is_file():
        ckpt = torch.load(best, map_location=device, weights_only=False)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        print(f"[stage2] init backbone from {best}")
    elif final.is_file():
        state = torch.load(final, map_location=device, weights_only=True)
        print(f"[stage2] init backbone from {final} (no best_val)")
    else:
        raise FileNotFoundError(f"No Stage1 weights under {s1_dir}")
    return {k.replace("backbone.", "", 1): v for k, v in state.items() if k.startswith("backbone.")}


@torch.no_grad()
def eval_checkpoint_on_test(cfg: ArmConfig, ckpt_path: Path, device: str, batch_size: int) -> dict:
    if cfg.stage == "stage1":
        model = SingleViewEffNetClassifier(
            pretrained=False,
            dense_units=STAGE4_DENSE_UNITS,
            dropout1=STAGE4_DROPOUT1,
            dropout2=STAGE4_DROPOUT2,
            use_grad_checkpoint=False,
        ).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        model.load_state_dict(state)
        ds = SingleViewMammogramDataset(
            SINGLE_DATA_DIR / "test.jsonl", cfg.height, cfg.width, train=False,
        )
        loader = DataLoader(
            ds, batch_size=batch_size, shuffle=False,
            collate_fn=single_view_collate_fn, num_workers=STAGE4_NUM_WORKERS,
        )
        metrics = evaluate_single(model, loader, device, [])
        out = {"test_metrics": metrics, "checkpoint": ckpt_path.name}
        if isinstance(ckpt, dict):
            out["ckpt_phase"] = ckpt.get("phase")
            out["ckpt_epoch"] = ckpt.get("epoch")
        del model
        torch.cuda.empty_cache()
        return out

    model = SharedTwoViewClassifier(
        pretrained=False,
        n_fusion_blocks=STAGE4_N_FUSION_BLOCKS,
        dense_units=STAGE4_DENSE_UNITS,
        dropout1=STAGE4_DROPOUT1,
        dropout2=STAGE4_DROPOUT2,
        use_grad_checkpoint=False,
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state)
    ds = PairedMammogramDataset(
        PAIRED_DATA_DIR / "test.jsonl", cfg.height, cfg.width, train=False,
    )
    loader = DataLoader(
        ds, batch_size=max(1, batch_size), shuffle=False,
        collate_fn=paired_collate_fn, num_workers=STAGE4_NUM_WORKERS,
    )
    metrics = evaluate_shared(model, loader, device, [])
    tta_scores, labels = [], []
    model.eval()
    for batch in loader:
        cc = batch["cc_pixel_values"].to(device)
        mlo = batch["mlo_pixel_values"].to(device)
        logits = model(cc, mlo)
        logits_f = model(torch.flip(cc, dims=[3]), torch.flip(mlo, dims=[3]))
        prob = 0.5 * (torch.sigmoid(logits) + torch.sigmoid(logits_f))
        tta_scores.extend(prob.float().cpu().tolist())
        labels.extend(batch["labels"].tolist())
    tta = compute_metrics(labels, tta_scores)
    out = {
        "test_metrics": metrics,
        "tta_metrics": tta,
        "checkpoint": ckpt_path.name,
    }
    if isinstance(ckpt, dict):
        out["ckpt_phase"] = ckpt.get("phase")
        out["ckpt_epoch"] = ckpt.get("epoch")
    del model
    torch.cuda.empty_cache()
    return out


def train_arm(
    cfg: ArmConfig,
    *,
    force_restart: bool = False,
    dry_run_epochs: int | None = None,
    use_amp: bool = True,
    device: str = "cuda",
) -> dict:
    # train one arm and return the best_val test metrics
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    torch.cuda.manual_seed_all(cfg.seed)

    output_dir = cfg.out_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    two_view = cfg.stage == "stage2"
    candidates = (4, 2, 1) if not two_view else (2, 1)
    batch_size = choose_batch_size(
        cfg.height, cfg.width, two_view=two_view, use_amp=use_amp,
        candidates=candidates, device=device,
    )
    accum_steps = max(1, cfg.target_effective_batch // batch_size)
    phase1_epochs = cfg.phase1_epochs
    phase2_epochs = cfg.phase2_epochs
    if dry_run_epochs is not None:
        phase1_epochs = min(phase1_epochs, dry_run_epochs)
        phase2_epochs = min(phase2_epochs, dry_run_epochs)

    run_config = _run_config_dict(cfg, batch_size, accum_steps, use_amp)
    run_config["phase1_epochs"] = phase1_epochs
    run_config["phase2_epochs"] = phase2_epochs
    with open(output_dir / "arm_config.json", "w") as f:
        json.dump({**cfg.to_dict(), "run_config": run_config}, f, indent=2)

    resume = resolve_resume(output_dir, run_config, force_restart=force_restart)
    if resume and resume["status"] == "done":
        print(f"[{cfg.arm_id}] already complete - skipping train")
        with open(output_dir / "test_results.json") as f:
            tr = json.load(f)
        return {
            "arm_id": cfg.arm_id,
            "status": "already_done",
            "test_auc_best_val": tr.get("best_val", {}).get("test_metrics", {}).get("auc_roc"),
            "test_auc_last": tr.get("last_epoch", {}).get("test_metrics", {}).get("auc_roc"),
            "out_dir": str(output_dir),
        }

    print(
        f"\n======== {cfg.arm_id} | {cfg.stage} | {cfg.height}x{cfg.width} | "
        f"p1={phase1_epochs}@{cfg.phase1_lr:.0e} p2={phase2_epochs}@{cfg.phase2_lr:.0e} | "
        f"aug={cfg.augment} bs={batch_size} accum={accum_steps} ckpt={cfg.use_grad_checkpoint} "
        f"========\n",
        flush=True,
    )

    if cfg.stage == "stage1":
        train_ds = SingleViewMammogramDataset(
            SINGLE_DATA_DIR / "train.jsonl", cfg.height, cfg.width,
            train=True, augment=cfg.augment,
        )
        val_ds = SingleViewMammogramDataset(
            SINGLE_DATA_DIR / "val.jsonl", cfg.height, cfg.width, train=False,
        )
        collate = single_view_collate_fn
        mode = "single"
        model = SingleViewEffNetClassifier(
            pretrained=True,
            dense_units=STAGE4_DENSE_UNITS,
            dropout1=STAGE4_DROPOUT1,
            dropout2=STAGE4_DROPOUT2,
            use_grad_checkpoint=cfg.use_grad_checkpoint,
        ).to(device)
    else:
        if not cfg.stage1_source:
            raise ValueError(f"{cfg.arm_id}: stage2 requires stage1_source")
        train_ds = PairedMammogramDataset(
            PAIRED_DATA_DIR / "train.jsonl", cfg.height, cfg.width,
            train=True, augment=cfg.augment,
        )
        val_ds = PairedMammogramDataset(
            PAIRED_DATA_DIR / "val.jsonl", cfg.height, cfg.width, train=False,
        )
        collate = paired_collate_fn
        mode = "shared"
        model = SharedTwoViewClassifier(
            pretrained=True,
            n_fusion_blocks=STAGE4_N_FUSION_BLOCKS,
            dense_units=STAGE4_DENSE_UNITS,
            dropout1=STAGE4_DROPOUT1,
            dropout2=STAGE4_DROPOUT2,
            use_grad_checkpoint=cfg.use_grad_checkpoint,
        ).to(device)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate,
        num_workers=STAGE4_NUM_WORKERS, drop_last=True, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate,
        num_workers=STAGE4_NUM_WORKERS, pin_memory=True,
    )
    pos_weight = compute_pos_weight(
        [1.0 if r["pathology"] == "malignant" else 0.0 for r in train_ds.rows]
    )
    print(f"train={len(train_ds)} val={len(val_ds)} pos_weight={pos_weight:.3f}", flush=True)

    history: list = []
    start_phase = "phase1"
    start_epoch = 1
    global_best = float("-inf")
    optimizer = scheduler = scaler = None

    # Stage2: load Stage1 backbone unless resuming mid-run
    if cfg.stage == "stage2" and not (resume and resume["status"] in ("resume", "phase1_done")):
        backbone_sd = _load_stage1_best_backbone(cfg.stage1_source, device)
        model.load_stage1_backbone(backbone_sd)

    if resume and resume["status"] in ("resume", "phase1_done"):
        ckpt = resume["ckpt"]
        model.load_state_dict(ckpt["model"])
        history = ckpt.get("history") or load_history_jsonl(output_dir / "history.jsonl")
        global_best = float(ckpt.get("global_best_auc", float("-inf")))
        # recompute global best from history if missing
        if history:
            global_best = max(
                global_best,
                max((h.get("auc_roc", float("-inf")) for h in history), default=float("-inf")),
            )
        phase, epoch = ckpt["phase"], ckpt["epoch"]
        print(f"[{cfg.arm_id}] resume {resume['source']}: {phase} e{epoch} "
              f"global_best={global_best:.4f}", flush=True)
        if resume["status"] == "phase1_done" or (phase == "phase1" and epoch >= phase1_epochs):
            start_phase, start_epoch = "phase2", 1
        elif phase == "phase1":
            start_phase, start_epoch = "phase1", epoch + 1
        else:
            start_phase, start_epoch = "phase2", epoch + 1

    log_prefix = cfg.arm_id

    # phase 1 trains the head with the backbone frozen
    if start_phase == "phase1":
        if cfg.stage == "stage1":
            for p in model.backbone.parameters():
                p.requires_grad = False
            frozen = [model.backbone]
        else:
            for p in model.shared_core.parameters():
                p.requires_grad = False
            frozen = [model.shared_core]
        trainable = [p for p in model.parameters() if p.requires_grad]
        history, optimizer, scheduler, scaler, global_best = train_phase(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            epochs=phase1_epochs,
            lr=cfg.phase1_lr,
            pos_weight=pos_weight,
            trainable_params=trainable,
            frozen_submodules=frozen,
            log_prefix=log_prefix,
            output_dir=output_dir,
            phase="phase1",
            run_config=run_config,
            start_epoch=start_epoch,
            history_so_far=history,
            global_best_auc=global_best,
            weight_decay=cfg.weight_decay,
            accum_steps=accum_steps,
            use_amp=use_amp,
            mode=mode,
        )
        atomic_torch_save(
            {
                "model": model.state_dict(),
                "phase": "phase1",
                "epoch": phase1_epochs,
                "history": history,
                "config": run_config,
                "global_best_auc": global_best,
                "optimizer": None,
                "scheduler": None,
                "scaler": None,
            },
            output_dir / "phase1_done.pt",
        )
        start_phase, start_epoch = "phase2", 1
        optimizer = scheduler = scaler = None

    # phase 2 fine-tunes the whole network
    if start_phase == "phase2" and phase2_epochs > 0 and start_epoch <= phase2_epochs:
        for p in model.parameters():
            p.requires_grad = True
        frozen = []
        trainable = list(model.parameters())
        history, optimizer, scheduler, scaler, global_best = train_phase(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            epochs=phase2_epochs,
            lr=cfg.phase2_lr,
            pos_weight=pos_weight,
            trainable_params=trainable,
            frozen_submodules=frozen,
            log_prefix=log_prefix,
            output_dir=output_dir,
            phase="phase2",
            run_config=run_config,
            start_epoch=start_epoch,
            history_so_far=history,
            global_best_auc=global_best,
            weight_decay=cfg.weight_decay,
            accum_steps=accum_steps,
            use_amp=use_amp,
            mode=mode,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

    # model.pt is the last epoch, then the run summary
    atomic_torch_save(model.state_dict(), output_dir / "model.pt")
    results = {
        "arm_id": cfg.arm_id,
        "config": run_config,
        "phase1_history": [h for h in history if h.get("phase") == "phase1"],
        "phase2_history": [h for h in history if h.get("phase") == "phase2"],
        "global_best_val_auc": global_best,
    }
    with open(output_dir / "results.json", "w") as f:
        json.dump(results, f, indent=2)

    # score the last epoch and best_val on the test set
    last_test = eval_checkpoint_on_test(cfg, output_dir / "model.pt", device, batch_size)
    best_path = output_dir / "best_val.pt"
    if best_path.is_file():
        best_test = eval_checkpoint_on_test(cfg, best_path, device, batch_size)
    else:
        best_test = last_test
        print(f"[{cfg.arm_id}] WARNING: no best_val.pt - using last epoch", flush=True)

    test_out = {
        "arm_id": cfg.arm_id,
        "last_epoch": last_test,
        "best_val": best_test,
        "archived_stage1_budget_auc": ARCHIVED_STAGE1_BUDGET_AUC,
        "archived_stage2_budget_auc": ARCHIVED_STAGE2_BUDGET_AUC,
        "archived_stage2_budget_tta_auc": ARCHIVED_STAGE2_BUDGET_TTA_AUC,
        "global_best_val_auc": global_best,
    }
    with open(output_dir / "test_results.json", "w") as f:
        json.dump(test_out, f, indent=2)

    last_auc = last_test["test_metrics"].get("auc_roc", float("nan"))
    best_auc = best_test["test_metrics"].get("auc_roc", float("nan"))
    tta_auc = None
    if "tta_metrics" in best_test:
        tta_auc = best_test["tta_metrics"].get("auc_roc")
    print(
        f"[{cfg.arm_id}] TEST last={last_auc:.4f} best_val={best_auc:.4f}"
        + (f" best_val+TTA={tta_auc:.4f}" if tta_auc is not None else "")
        + f" | val_best={global_best:.4f}"
        + f" | beat_s1_budget={best_auc > ARCHIVED_STAGE1_BUDGET_AUC if cfg.stage == 'stage1' else 'n/a'}",
        flush=True,
    )

    del model
    torch.cuda.empty_cache()
    return {
        "arm_id": cfg.arm_id,
        "status": "completed",
        "stage": cfg.stage,
        "height": cfg.height,
        "width": cfg.width,
        "phase2_lr": cfg.phase2_lr,
        "augment": cfg.augment,
        "val_best_auc": global_best,
        "test_auc_last": last_auc,
        "test_auc_best_val": best_auc,
        "test_auc_best_val_tta": tta_auc,
        "out_dir": str(output_dir),
        "config": run_config,
    }


def run_arm_with_fallbacks(
    cfg: ArmConfig,
    *,
    force_restart: bool = False,
    dry_run_epochs: int | None = None,
    use_amp: bool = True,
    device: str = "cuda",
    allow_fallback: bool = True,
) -> dict:
    chain = [cfg] + (oom_fallback_variants(cfg) if allow_fallback else [])
    errors = []
    for i, variant in enumerate(chain):
        try:
            return train_arm(
                variant,
                force_restart=force_restart and i == 0,
                dry_run_epochs=dry_run_epochs,
                use_amp=use_amp,
                device=device,
            )
        except RuntimeError as e:
            msg = str(e)
            print(f"[fallback] {variant.arm_id} failed: {msg}", flush=True)
            errors.append({"arm_id": variant.arm_id, "error": msg})
            torch.cuda.empty_cache()
            if "out of memory" not in msg.lower() and i == 0 and not allow_fallback:
                raise
            # a failed first try still moves on to the smaller setup
            continue
        except Exception as e:
            print(f"[error] {variant.arm_id}: {e}\n{traceback.format_exc()}", flush=True)
            errors.append({"arm_id": variant.arm_id, "error": str(e)})
            torch.cuda.empty_cache()
            continue
    return {
        "arm_id": cfg.arm_id,
        "status": "failed",
        "errors": errors,
        "test_auc_best_val": None,
    }


# stage 1 search, then stage 2 from the best of those


def pick_stage2_sources(n: int = 2) -> list[str]:
    # stage 1 arms that finished, highest test AUC first
    if not LEADERBOARD_PATH.is_file():
        return []
    with open(LEADERBOARD_PATH) as f:
        board = json.load(f)
    s1 = [
        r for r in board
        if r.get("stage") == "stage1" and r.get("status") in ("completed", "already_done")
        and r.get("test_auc_best_val") is not None
    ]
    s1.sort(key=lambda r: r["test_auc_best_val"], reverse=True)
    return [r["arm_id"] for r in s1[:n]]


def find_arm(arm_id: str, queue: list[ArmConfig]) -> ArmConfig | None:
    for a in queue:
        if a.arm_id == arm_id:
            return a
    # the arm might only be saved on disk
    cfg_path = SWEEP_ROOT / arm_id / "arm_config.json"
    if cfg_path.is_file():
        with open(cfg_path) as f:
            raw = json.load(f)
        keys = {f.name for f in ArmConfig.__dataclass_fields__.values()}
        return ArmConfig(**{k: raw[k] for k in keys if k in raw})
    return None


def run_sweep(
    *,
    max_hours: float = 24.0,
    top_k_stage2: int = 2,
    force_restart: bool = False,
    dry_run_epochs: int | None = None,
    use_amp: bool = True,
    allow_fallback: bool = True,
    stage1_only: bool = False,
    arm_filter: list[str] | None = None,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda":
        raise RuntimeError("CUDA required for the direct-classifier sweep")

    SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
    status = load_status()
    now = time.time()
    if status.get("started_at") is None:
        status["started_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
        status["start_ts"] = now
    # a resumed run keeps the deadline it already has
    if status.get("deadline_ts") is None or force_restart:
        if status.get("deadline_ts") is None:
            status["deadline_ts"] = now + max_hours * 3600
    # before any arm finishes, the time limit starts from now
    if not status.get("completed") and not status.get("failed"):
        status["deadline_ts"] = now + max_hours * 3600

    deadline = float(status["deadline_ts"])
    print(
        f"Chollet HP sweep | budget until {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(deadline))} "
        f"({max(0, deadline - now) / 3600:.1f}h left) | root={SWEEP_ROOT}",
        flush=True,
    )

    stage1_queue = build_stage1_queue()
    if arm_filter:
        stage1_queue = [a for a in stage1_queue if a.arm_id in arm_filter]
    if dry_run_epochs is not None:
        print(f"SMOKE: dry_run_epochs={dry_run_epochs}", flush=True)

    def time_left() -> float:
        return deadline - time.time()

    def should_stop(est_hours: float) -> bool:
        # stop if under 10 minutes are left, or under half the estimate
        return time_left() < max(600.0, est_hours * 3600 * 0.5)

    # stage 1 arms
    for cfg in stage1_queue:
        if cfg.arm_id in status.get("completed", []):
            print(f"[skip] {cfg.arm_id} already in completed list", flush=True)
            continue
        # also skip if results on disk
        if (cfg.out_dir / "test_results.json").is_file() and not force_restart:
            if cfg.arm_id not in status.get("completed", []):
                status.setdefault("completed", []).append(cfg.arm_id)
                save_status(status)
            print(f"[skip] {cfg.arm_id} has test_results.json", flush=True)
            continue
        if should_stop(cfg.est_hours):
            print(f"[time] skip {cfg.arm_id} (est {cfg.est_hours}h, left {time_left()/3600:.2f}h)", flush=True)
            status.setdefault("skipped", []).append(
                {"arm_id": cfg.arm_id, "reason": "time_budget"}
            )
            save_status(status)
            continue

        status["active"] = cfg.arm_id
        save_status(status)
        t0 = time.time()
        summary = run_arm_with_fallbacks(
            cfg,
            force_restart=force_restart,
            dry_run_epochs=dry_run_epochs,
            use_amp=use_amp,
            device=device,
            allow_fallback=allow_fallback,
        )
        summary["wall_hours"] = (time.time() - t0) / 3600
        if summary.get("status") == "failed":
            status.setdefault("failed", []).append(summary)
        else:
            status.setdefault("completed", []).append(summary["arm_id"])
            update_leaderboard(summary)
        status["active"] = None
        save_status(status)
        print(f"[status] completed={len(status.get('completed', []))} "
              f"failed={len(status.get('failed', []))} "
              f"time_left={time_left()/3600:.2f}h", flush=True)

    if stage1_only:
        print("stage1_only - done", flush=True)
        _write_final_summary(status)
        return status

    # stage 2, from the strongest stage 1 arms
    sources = pick_stage2_sources(n=top_k_stage2)
    status["stage2_planned"] = sources
    save_status(status)
    print(f"\n[stage2] promoting top-{top_k_stage2} Stage1 arms: {sources}", flush=True)

    for rank, s1_id in enumerate(sources):
        src = find_arm(s1_id, stage1_queue)
        if src is None:
            print(f"[stage2] cannot resolve config for {s1_id}", flush=True)
            continue
        s2 = make_stage2_arm(src, rank=rank)
        if (s2.out_dir / "test_results.json").is_file() and not force_restart:
            print(f"[skip] {s2.arm_id} already done", flush=True)
            if s2.arm_id not in status.get("completed", []):
                status.setdefault("completed", []).append(s2.arm_id)
            continue
        if should_stop(s2.est_hours):
            print(f"[time] skip {s2.arm_id}", flush=True)
            status.setdefault("skipped", []).append(
                {"arm_id": s2.arm_id, "reason": "time_budget"}
            )
            save_status(status)
            continue
        status["active"] = s2.arm_id
        save_status(status)
        t0 = time.time()
        summary = run_arm_with_fallbacks(
            s2,
            force_restart=force_restart,
            dry_run_epochs=dry_run_epochs,
            use_amp=use_amp,
            device=device,
            allow_fallback=allow_fallback,
        )
        summary["wall_hours"] = (time.time() - t0) / 3600
        if summary.get("status") == "failed":
            status.setdefault("failed", []).append(summary)
        else:
            status.setdefault("completed", []).append(summary["arm_id"])
            update_leaderboard(summary)
        status["active"] = None
        save_status(status)

    _write_final_summary(status)
    return status


def _write_final_summary(status: dict):
    summary_path = SWEEP_ROOT / "final_summary.json"
    board = []
    if LEADERBOARD_PATH.is_file():
        with open(LEADERBOARD_PATH) as f:
            board = json.load(f)
    payload = {
        "status": status,
        "leaderboard": board,
        "archived_baselines": {
            "stage1_budget": ARCHIVED_STAGE1_BUDGET_AUC,
            "stage2_budget": ARCHIVED_STAGE2_BUDGET_AUC,
            "stage2_budget_tta": ARCHIVED_STAGE2_BUDGET_TTA_AUC,
        },
        "written_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(summary_path, "w") as f:
        json.dump(payload, f, indent=2)
    print("\nBest validation checkpoints, scored on the test set:", flush=True)
    for i, r in enumerate(board[:15], 1):
        print(
            f"{i:2d}. {r.get('arm_id', '?'):40s}  "
            f"test_best={r.get('test_auc_best_val')}  "
            f"test_last={r.get('test_auc_last')}  "
            f"val={r.get('val_best_auc')}  "
            f"tta={r.get('test_auc_best_val_tta')}",
            flush=True,
        )
    print(f"Wrote {summary_path}", flush=True)


print("Stage 1 search queue (not training):")
for arm in build_stage1_queue():
    print(f"  {arm.arm_id}  p1={arm.phase1_epochs}@{arm.phase1_lr:.0e}  p2={arm.phase2_epochs}@{arm.phase2_lr:.0e}  aug={arm.augment}")
print("Arm I kept: s1_h_p2_3e5 -> s2_from_s1_h_p2_3e5")

if RUN_WINNING_ARM_TRAIN:
    run_sweep(max_hours=6.0, top_k_stage2=1, arm_filter=["s1_h_p2_3e5"])
elif RUN_SWEEP:
    run_sweep(max_hours=24.0, top_k_stage2=2)
else:
    print("I left RUN_SWEEP and RUN_WINNING_ARM_TRAIN off. The checkpoint stays in weights/direct_classifier/.")


Stage 1 search queue (not training):
  s1_h_ctrl_p2_1e5  p1=5@1e-03  p2=20@1e-05  aug=light
  s1_h_p2_3e5  p1=5@1e-03  p2=20@3e-05  aug=light
  s1_h_p2_1e4  p1=5@1e-03  p2=20@1e-04  aug=light
  s1_h_p2_3e5_medaug  p1=5@1e-03  p2=20@3e-05  aug=medium
  s1_h_long_p2_3e5  p1=5@1e-03  p2=35@3e-05  aug=light
  s1_h_p1_10_p2_3e5  p1=10@1e-03  p2=20@3e-05  aug=light
Arm I kept: s1_h_p2_3e5 -> s2_from_s1_h_p2_3e5
I left RUN_SWEEP and RUN_WINNING_ARM_TRAIN off. The checkpoint stays in weights/direct_classifier/.


## Inference and GradCAM

I keep the checkpoint with the best validation AUC across phase 1 and phase 2. When a better score comes in, best_val.pt is written to a temporary file and then moved over the old file, so a crash during the write does not wipe the best weights. last.pt is the latest epoch, overwritten every epoch. At the end of a run I score both the last epoch and best_val on the test set. I use the best_val weights for both designs, because validation AUC peaked in phase 1.

At test time I run the pair and a horizontally flipped copy, then average the two probabilities. That moved the direct classifier from 0.822 to 0.830, and the patch classifier from 0.850 to 0.851.

![Test time augmentation](figures/tta.png)

Test time augmentation diagram

At test time both designs do the same thing. The full CC and MLO views are normalized with the ImageNet constants, resized to 576x448, and passed through SharedTwoView. One sigmoid is applied to the one logit. There is no crop and no patch head.


In [5]:

if _SMOKE:
    smoke_model = make_stage2_model(pretrained=False, device="cpu")
    smoke_ckpt = load_checkpoint(FROZEN_DC_S2, smoke_model, "cpu")
    smoke_logit = smoke_model(torch.zeros(1, 3, HEIGHT, WIDTH), torch.zeros(1, 3, HEIGHT, WIDTH))
    if tuple(smoke_logit.shape) != (1,):
        raise RuntimeError(f"expected one logit, got {tuple(smoke_logit.shape)}")
    print("smoke load ok", smoke_ckpt.get("phase") if isinstance(smoke_ckpt, dict) else type(smoke_ckpt))
elif RUN_EVAL:
    direct_eval = score_frozen_direct_classifier()
    if abs(direct_eval["tta_metrics"]["auc_roc"] - ARCHIVED_TTA) > 0.01:
        raise RuntimeError(
            f"TTA {direct_eval['tta_metrics']['auc_roc']} is more than 0.01 from archived {ARCHIVED_TTA}"
        )
else:
    print("RUN_EVAL is off")


Direct Classifier JPEG n=291 plain=0.8221 TTA=0.8300 (archived 0.8222 / 0.8301)
